# Taller 02 — Cadenas de Markov en tiempo discreto

**Procesos Estocásticos · Área de Modelado y Simulación · Universidad EIA**

---

Este taller cubre los temas de las primeras tres semanas: matriz de transición, ecuación de Chapman-Kolmogorov, clasificación de estados y distribución estacionaria.

## Contexto: gestión de inventario

Una bodega clasifica su nivel de inventario de un producto en tres estados al cierre de cada semana:

- **Estado 0 — Alto**
- **Estado 1 — Medio**
- **Estado 2 — Bajo**

El comportamiento de una semana a la siguiente se modela con la matriz de transición

$$
P = \begin{pmatrix}
0.5 & 0.4 & 0.1 \\
0.2 & 0.5 & 0.3 \\
0.3 & 0.3 & 0.4
\end{pmatrix}
$$

donde $P_{ij}$ es la probabilidad de pasar del nivel $i$ al nivel $j$ en una semana.

## Reglas

- Escribe tu solución donde se indica, eliminando la línea `raise NotImplementedError()`.
- No modifiques las celdas de prueba ni los nombres de las variables solicitadas.
- Antes de entregar: *Kernel → Restart Kernel and Run All Cells*, verifica que corre completo, descarga el `.ipynb` y renómbralo `<tu_código>_taller02.ipynb`.
- Recuerda: el botón **Validate** ejecuta solo las pruebas visibles; la calificación incluye pruebas ocultas.

In [19]:
import numpy as np
from numpy.linalg import matrix_power

np.set_printoptions(precision=4, suppress=True)

estados = ["Alto", "Medio", "Bajo"]

---

## Ejercicio 1 — Matriz de transición (2 puntos)

Define la matriz de transición del enunciado como un arreglo de NumPy llamado `P`.

In [20]:
P = np.array([[0.5,0.4,0.1],
              [0.2,0.5,0.3],
              [0.3,0.3,0.4]])

In [21]:
# PRUEBA VISIBLE
assert P.shape == (3, 3), 'P debe ser 3x3.'
assert np.allclose(P.sum(axis=1), 1.0), 'Cada fila debe sumar 1.'
print('Prueba visible superada.')

Prueba visible superada.


---

## Ejercicio 2 — Chapman-Kolmogorov (2 puntos)

Usando la ecuación de Chapman-Kolmogorov, calcula la probabilidad de que, partiendo de inventario **Alto**, el sistema esté en nivel **Bajo** después de **3 semanas**.

Guarda ese único valor escalar en la variable `p_alto_bajo_3`.

*Pista:* la probabilidad a $n$ pasos está en la matriz $P^n$. Usa `matrix_power`.

In [22]:
P3 = matrix_power(P,3)
p_alto_bajo_3 = P3[0,2]

print(f'P(Bajo en 3 semanas | Alto hoy) = {p_alto_bajo_3:.4f}')

P(Bajo en 3 semanas | Alto hoy) = 0.2490


In [23]:
# PRUEBA VISIBLE
assert np.isscalar(p_alto_bajo_3) or p_alto_bajo_3.ndim == 0, 'Debe ser un escalar.'
assert 0 <= p_alto_bajo_3 <= 1, 'Debe ser una probabilidad entre 0 y 1.'
print('Prueba visible superada.')

Prueba visible superada.


---

## Ejercicio 3 — Distribución a n pasos (2 puntos)

Si esta semana el inventario está en **Alto**, ¿cuál es la distribución de probabilidad sobre los tres niveles **dentro de 2 semanas**?

Define la distribución inicial `pi_0` (como vector fila que representa "Alto" con certeza) y calcula `pi_2`, la distribución a 2 pasos.

*Pista:* $\pi_n = \pi_0 P^n$.

In [24]:
pi_0 = np.array([1,0,0])
P2 = matrix_power(P,2)
pi_2 = pi_0 @ P2

for estado, p in zip(estados, pi_2):
    print(f'{estado:6}: {p:.4f}')

Alto  : 0.3600
Medio : 0.4300
Bajo  : 0.2100


In [25]:
# PRUEBA VISIBLE
assert pi_2.shape == (3,), 'pi_2 debe ser un vector de longitud 3.'
assert np.isclose(pi_2.sum(), 1.0), 'pi_2 debe sumar 1.'
print('Prueba visible superada.')

Prueba visible superada.


---

## Ejercicio 4 — Distribución estacionaria (3 puntos)

Completa la función `distribucion_estacionaria(P)` que resuelve $\pi = \pi P$ con la condición de normalización $\sum_i \pi_i = 1$, y aplícala a la matriz del inventario. Guarda el resultado en `pi_est`.

*Pista:* plantea el sistema $(P^T - I)\,\pi = 0$ y reemplaza la última ecuación por la de normalización.

In [26]:
def distribucion_estacionaria(P):
    A = P.T - np.eye(3)
    b = np.zeros(3)

    A[-1, :] = 1
    b[-1] = 1

    pi = np.linalg.solve(A, b)

    return pi

pi_est = distribucion_estacionaria(P)
for estado, p in zip(estados, pi_est):
    print(f'{estado:6}: {p:.4f}')

Alto  : 0.3231
Medio : 0.4154
Bajo  : 0.2615


In [27]:
# PRUEBA VISIBLE
assert pi_est.shape == (3,), 'pi_est debe tener longitud 3.'
assert np.isclose(pi_est.sum(), 1.0), 'pi_est debe sumar 1.'
assert np.allclose(pi_est @ P, pi_est), 'pi_est debe cumplir pi = pi P.'
print('Prueba visible superada.')

Prueba visible superada.


---

## Ejercicio 5 — Clasificación de la cadena (2 puntos)

Analiza la cadena del inventario y responde con valores booleanos (`True` / `False`):

- `es_irreducible`: ¿todos los estados se comunican entre sí?
- `es_aperiodica`: ¿la cadena es aperiódica?

Puedes razonarlo o calcularlo con código. *Pista para irreducibilidad:* revisa si $(I+P)^{n-1}$ tiene todas sus entradas positivas. *Pista para aperiodicidad:* basta con que algún estado tenga probabilidad de permanecer en sí mismo.

In [28]:
# TU CÓDIGO AQUÍ — elimina esta línea y la siguiente
def es_irreducible(P):
    n = P.shape[0]
    I = np.eye(n)

    matrizAlcanzabilidad = np.linalg.matrix_power(I + P, n - 1) #Saber si un estado es alcanzable desde otro estado

    return bool(np.all(matrizAlcanzabilidad > 0))

def es_aperiodica(P):
    return bool(np.any(np.diag(P) > 0))

es_irreducible = es_irreducible(P)
es_aperiodica = es_aperiodica(P)
print('Irreducible:', es_irreducible)
print('Aperiódica: ', es_aperiodica)

Irreducible: True
Aperiódica:  True


In [29]:
# PRUEBA VISIBLE
assert isinstance(es_irreducible, bool), 'es_irreducible debe ser booleano.'
assert isinstance(es_aperiodica, bool), 'es_aperiodica debe ser booleano.'
print('Prueba visible superada.')

Prueba visible superada.


---

## Ejercicio 6 — Interpretación: tiempo medio de retorno (2 puntos)

Para una cadena irreducible y aperiódica, el **tiempo medio de retorno** a un estado $i$ es $m_i = 1/\pi_i$, donde $\pi_i$ es su probabilidad estacionaria.

Calcula el tiempo medio de retorno al estado **Alto** (en semanas) y guárdalo en `tiempo_retorno_alto`.

Interpreta: en promedio, ¿cada cuántas semanas la bodega vuelve a tener inventario Alto?

In [30]:
tiempo_retorno_alto = 1/pi_est[0]

print(f'Tiempo medio de retorno a Alto: {tiempo_retorno_alto:.2f} semanas')

Tiempo medio de retorno a Alto: 3.10 semanas


In [31]:
# PRUEBA VISIBLE
assert tiempo_retorno_alto > 0, 'El tiempo de retorno debe ser positivo.'
print('Prueba visible superada.')

Prueba visible superada.


---

## Cierre

Verifica que el notebook corre completo con *Kernel → Restart Kernel and Run All Cells*, descárgalo, renómbralo `<tu_código>_taller02.ipynb` y súbelo a Moodle.

**Total: 13 puntos.**